In [54]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

In [55]:
x, y = fetch_california_housing(return_X_y=True, as_frame=True)

x.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [56]:
x_train, x_testval, y_train, y_testval = train_test_split(x, y, test_size=0.2, random_state=42)
x_test, x_val, y_test, y_val = train_test_split(x_testval, y_testval, test_size=0.5, random_state=42)

In [57]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [58]:
lat_lon_cols = ['Latitude', 'Longitude']
processed_cols = [c for c in x_train.columns if c not in lat_lon_cols]

Q1 = x_train[processed_cols].quantile(0.25)
Q3 = x_train[processed_cols].quantile(0.75)
IQR = Q3 - Q1

lower_bound = torch.tensor((Q1 - 1.5 * IQR).values, dtype=torch.float32)
upper_bound = torch.tensor((Q3 + 1.5 * IQR).values, dtype=torch.float32)
mean = torch.tensor(x_train[processed_cols].mean().values, dtype=torch.float32)
std = torch.tensor(x_train[processed_cols].std().values, dtype=torch.float32)

In [59]:
class ScalingLayer(nn.Module):
    def __init__(self, mean: torch.Tensor, std: torch.Tensor):
        super().__init__()
        self.register_buffer('mean', mean)
        self.register_buffer('std', std)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        return (X - self.mean) / (self.std + 1e-8)

In [60]:
class PreprocessingLayer(nn.Module):
    processed_cols_idx = [0, 1, 2, 3, 4, 5]
    latlon_idx = [6, 7]

    def __init__(self, mean, std, lower, upper):
        super().__init__()
        self.scaler = ScalingLayer(mean, std)
        self.register_buffer('lower', lower)
        self.register_buffer('upper', upper)

    def forward(self, x):                                      
        x_main = x[:, self.processed_cols_idx]
        x_latlon = x[:, self.latlon_idx]
        x_main = torch.clamp(x_main, self.lower, self.upper) 
        x_main = self.scaler(x_main)                         
        return torch.cat([x_main, x_latlon], dim=1)

In [61]:
class HousingDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).to(device)
        self.y = torch.tensor(y, dtype=torch.float32).to(device)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [62]:
train_data = HousingDataset(x_train.values, y_train.values)
val_data = HousingDataset(x_val.values,   y_val.values)
test_data = HousingDataset(x_test.values,  y_test.values)

In [63]:
class HousingNetwork(nn.Module):
    def __init__(self, hidden1_size, hidden2_size, hidden3_size, mean, std, lower, upper):
        super().__init__()
        self.preprocessing = PreprocessingLayer(mean, std, lower, upper)

        self.network = nn.Sequential(
            nn.Linear(8, hidden1_size),   
            nn.ReLU(),
            nn.Linear(hidden1_size, hidden2_size),
            nn.ReLU(),
            nn.Linear(hidden2_size, hidden3_size),
            nn.ReLU(),
            nn.Linear(hidden3_size, 1)   
        )

    def forward(self, x):
        x = self.preprocessing(x)
        return self.network(x).squeeze(1)

In [64]:
def train(_model, _train_loader, _val_loader, _criterion, _optimizer, _num_epochs):
    train_losses = []
    val_losses   = []
    patience = 5
    min_delta = 1e-4
    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_weights = None

    for epoch in range(_num_epochs):
        _model.train()
        running_loss = 0.0

        for X_batch, y_batch in tqdm(_train_loader, desc=f"Epoch {epoch+1}/{_num_epochs}"):
            _optimizer.zero_grad()
            outputs = _model(X_batch)
            loss = _criterion(outputs, y_batch)
            loss.backward()
            _optimizer.step()
            running_loss += loss.item() * X_batch.size(0)

        epoch_train_loss = running_loss / len(_train_loader.dataset)

        _model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_val, y_val in _val_loader:
                val_outputs = _model(X_val)
                val_loss += _criterion(val_outputs, y_val).item() * X_val.size(0)

        epoch_val_loss = val_loss / len(_val_loader.dataset)

        train_losses.append(epoch_train_loss)
        val_losses.append(epoch_val_loss)

        print(f"Epoch {epoch+1} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")

        if epoch_val_loss < best_val_loss - min_delta:
            best_val_loss = epoch_val_loss
            best_weights = _model.state_dict()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                _model.load_state_dict(best_weights)
                break

    return train_losses, val_losses

In [65]:
criterion = nn.MSELoss()

In [ ]:
# Modelo 1. 128-64-32 Adam
model = HousingNetwork(
    hidden1_size=128, hidden2_size=64, hidden3_size=32,
    mean=mean, std=std, lower=lower_bound, upper=upper_bound
).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 50
batch_size = 64

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data,   batch_size=batch_size)
train_losses, val_losses = train(model, train_loader, val_loader, criterion, optimizer, num_epochs)

Epoch 1/50: 100%|██████████| 258/258 [00:00<00:00, 380.02it/s]


Epoch 1 | Train Loss: 1.3680 | Val Loss: 1.2724


Epoch 2/50: 100%|██████████| 258/258 [00:00<00:00, 450.76it/s]


Epoch 2 | Train Loss: 1.3172 | Val Loss: 1.2411


Epoch 3/50: 100%|██████████| 258/258 [00:00<00:00, 435.43it/s]


...

Epoch 26 | Train Loss: 0.9722 | Val Loss: 0.9981


Epoch 27/50: 100%|██████████| 258/258 [00:00<00:00, 416.49it/s]


Epoch 27 | Train Loss: 0.9130 | Val Loss: 0.9979


Epoch 28/50: 100%|██████████| 258/258 [00:00<00:00, 347.90it/s]


Epoch 28 | Train Loss: 0.8846 | Val Loss: 1.0760
Early stopping at epoch 28


In [ ]:
# Modelo 2: 64-64-32 Adam
model2 = HousingNetwork(
    hidden1_size=64, hidden2_size=64, hidden3_size=32,
    mean=mean, std=std, lower=lower_bound, upper=upper_bound
).to(device)

optimizer = optim.Adam(model2.parameters(), lr=0.001)
num_epochs = 50
batch_size = 64

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data,   batch_size=batch_size)
train_losses2, val_losses2 = train(model2, train_loader, val_loader, criterion, optimizer, num_epochs)

Epoch 1/50: 100%|██████████| 258/258 [00:00<00:00, 343.27it/s]


Epoch 1 | Train Loss: 1.3414 | Val Loss: 0.8431


Epoch 2/50: 100%|██████████| 258/258 [00:00<00:00, 349.16it/s]


Epoch 2 | Train Loss: 0.7169 | Val Loss: 0.6239


Epoch 3/50: 100%|██████████| 258/258 [00:00<00:00, 385.03it/s]


...

Epoch 33 | Train Loss: 0.4490 | Val Loss: 0.4584


Epoch 34/50: 100%|██████████| 258/258 [00:00<00:00, 397.86it/s]


Epoch 34 | Train Loss: 0.4476 | Val Loss: 0.4615


Epoch 35/50: 100%|██████████| 258/258 [00:00<00:00, 344.70it/s]

Epoch 35 | Train Loss: 0.4397 | Val Loss: 0.4526
Early stopping at epoch 35


In [ ]:
# Modelo 3: 256-128-64 SGD
model3 = HousingNetwork(
    hidden1_size=256, hidden2_size=128, hidden3_size=64,
    mean=mean, std=std, lower=lower_bound, upper=upper_bound
).to(device)
    
optimizer = optim.SGD(model3.parameters(), lr=0.0001)
num_epochs = 50
batch_size = 64

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data,   batch_size=batch_size)
train_losses, val_losses = train(model3, train_loader, val_loader, criterion, optimizer, num_epochs)